In [ ]:
import json
import re
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "self-instruct").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Cannot find the project root")


def is_truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


project_root = find_project_root()
legacy_mode = is_truthy(os.getenv("LEGACY_MODELS"))

if legacy_mode:
    input_file_name = "self_instructed_legacy_models_step_5.jsonl"
    output_file_name = "self_instructed_legacy_models_step_6.jsonl"
else:
    input_file_name = "self_instructed_models_step_5.jsonl"
    output_file_name = "self_instructed_models_step_6.jsonl"

file = project_root / "self-instruct" / "data" / input_file_name

data = []
with open(file, "r") as f:
     data = json.load(f)

print(f"LEGACY_MODELS={os.getenv('LEGACY_MODELS')} -> {input_file_name}")
print(f"Loaded {len(data)} entries from {file}")

# remove models if contains "None" string in modelcard or queries
final_cleaned_models = []
for entry in data:
    modelcard = entry.get("modelcard", "")
    queries = entry.get("queries", [])

    # Check if modelcard contains "None" or any query is exactly "None"
    if "None" in modelcard or any("None" in str(q) for q in queries):
        continue

    final_cleaned_models.append(entry)

print(f"Removed entries with 'None': {len(data) - len(final_cleaned_models)}")
print(f"Final cleaned models: {len(final_cleaned_models)}")
data = final_cleaned_models

In [ ]:
# Extract and validate queries from all entries
count_valid = 0
count_invalid_json = 0
count_invalid_structure = 0
count_manual_extraction = 0
failed_entries = []  # Store failed entries
cleaned_models = []  # Store valid entries

for idx, entry in enumerate(data):
    generated_str = entry.get("generated", "")

    # Remove text before </think> if present
    if "</think>" in generated_str:
        text_after_think = generated_str.split("</think>", 1)[-1].strip()
    else:
        text_after_think = generated_str

    # Try to find JSON object containing "queries" key
    match_json = re.search(
        r'\{[^}]*"queries"[^}]*:.*?\].*?\}', text_after_think, re.DOTALL
    )

    if match_json:
        json_str = match_json.group(0).strip()
        try:
            # Try to parse the JSON
            generated_dict = json.loads(json_str)

            # Extract queries
            queries = generated_dict.get("queries", [])

            # Validate: must be a list with 20 strings
            if (
                isinstance(queries, list)
                and len(queries) == 20
                and all(isinstance(q, str) for q in queries)
            ):
                count_valid += 1
                entry["queries"] = queries
                cleaned_models.append(entry)
            else:
                count_invalid_structure += 1
                failed_entries.append(entry)
        except json.JSONDecodeError as e:
            # If parsing fails, try to extract manually with regex
            # Look for array pattern: ["query1", "query2", ...]
            manual_match = re.search(
                r'"queries"\s*:\s*(\[.*?\])', text_after_think, re.DOTALL
            )

            if manual_match:
                try:
                    queries = json.loads(manual_match.group(1))

                    # Validate manually extracted queries
                    if (
                        isinstance(queries, list)
                        and len(queries) == 20
                        and all(isinstance(q, str) for q in queries)
                    ):
                        count_manual_extraction += 1
                        entry["queries"] = queries
                        cleaned_models.append(entry)
                    else:
                        count_invalid_structure += 1
                        failed_entries.append(entry)
                except json.JSONDecodeError:
                    count_invalid_json += 1
                    failed_entries.append(entry)
            else:
                count_invalid_json += 1
                failed_entries.append(entry)
    else:
        # No JSON found
        count_invalid_json += 1
        failed_entries.append(entry)

print(f"Processing complete:")
print(f"  ✅ Valid entries (direct): {count_valid}")
print(f"  ✅ Valid entries (manual extraction): {count_manual_extraction}")
print(f"  ✅ Total valid: {len(cleaned_models)}")
print(f"  ❌ Invalid JSON: {count_invalid_json}")
print(f"  ❌ Invalid structure: {count_invalid_structure}")
print(f"  📊 Total processed: {len(data)}")
print(f"  📈 Success rate: {len(cleaned_models) / len(data) * 100:.2f}%")

In [ ]:
# Print sample of failed entries
print(f"\n{'=' * 60}")
print(f"FAILED ENTRIES DETAILS ({len(failed_entries)} total)")
print(f"{'=' * 60}\n")

for i, entry in enumerate(failed_entries[:5], 1):  # Show first 5
    print(f"{i}. Model: {entry['model_id']}")
    if "raw_text" in entry:
        print(f"   Raw text: {entry['raw_text'][:150]}...")
    elif "queries" in entry:
        print(f"   Found {len(entry['queries'])} queries")
    print()

if len(failed_entries) > 5:
    print(f"... and {len(failed_entries) - 5} more failed entries\n")

In [ ]:
failed_entries


In [ ]:
# put failed entries are correct so put them in the cleaned_models list as well
for entry in failed_entries:
    entry["queries"] = entry.get("queries", [])
    cleaned_models.append(entry)

In [ ]:
# Print sample of valid entries
print(f"{'=' * 60}")
print(f"VALID ENTRIES SAMPLES ({len(cleaned_models)} total)")
print(f"{'=' * 60}\n")

for i, entry in enumerate(cleaned_models[:3], 1):  # Show first 3
    print(f"{i}. Model: {entry.get('model_id', 'unknown')}")
    print(f"   Number of queries: {len(entry['queries'])}")
    print(f"   First query: {entry['queries'][0][:100]}...")
    print(f"   Last query: {entry['queries'][-1][:100]}...")
    print()

In [ ]:
# duplicate models, one for each prompt in the queries list
final_prompts = []
for entry in cleaned_models:
    queries = entry["queries"]
    for q in queries:
        final_prompts.append(
            {
                "Instruction": q,
                "model_id": entry["model_id"],
                "description": entry["modelcard"],
                "domain": entry["domain"],
                "created_at": entry["created_at"],
            }
        )
print(f"Total final prompts generated: {len(final_prompts)}")
print(f"Sample final prompts:")
final_prompts[0]

In [ ]:
# compute instruction lenght statistics with pandas
import pandas as pd

df = pd.DataFrame(final_prompts)
print(f"Total instructions: {len(df)}")
print(f"Unique models: {df['model_id'].nunique()}")
print(f"Instruction length stats (chars):")
print(df["Instruction"].str.len().describe())

In [ ]:
# print queries with min length
min_len = df["Instruction"].str.len().min()
print(f"\nInstructions with minimum length ({min_len} chars):")
shortest_instructions = df[df["Instruction"].str.len() == min_len]
for i, row in shortest_instructions.iterrows():
    print(f"- Model: {row['model_id']}, Instruction: {row['Instruction']}")

In [ ]:
# sort by model_id and drop instruction duplicates
print(f"Total unique models: {df['model_id'].nunique()}")
print(f"Total instructions before dropping duplicates: {len(df)}")
df = df.sort_values(by=["model_id", "Instruction"])
df = df.drop_duplicates(subset=["Instruction"], keep="last")
print(f"Total instructions after dropping duplicates: {len(df)}")
# print total unique models after dropping duplicates
print(f"Total unique models after dropping duplicates: {df['model_id'].nunique()}")

In [ ]:
# group by domain and count instructions per domain
domain_counts = df["domain"].value_counts()
print(f"\nInstruction counts by domain:")
print(domain_counts)

In [ ]:
# save final prompts to jsonl file
output_file = project_root / "self-instruct" / "data" / output_file_name
output_file.parent.mkdir(parents=True, exist_ok=True)
df.to_json(output_file, orient="records", lines=True, force_ascii=False)